# Behavioral Finance Monte Carlo Simulator
## How investor psychology destroys long-run wealth

*A quantitative study using real SPY/EFA/AGG market data (2005–2024)*

---

### Abstract

This notebook simulates portfolio performance for three investor archetypes over
a 10-year horizon using 10,000 Monte Carlo paths. Return distributions are fitted
to **real historical data** using Student's t-distributions — capturing the fat
tails and negative skew absent from naïve Gaussian models.

The central finding: **behavioural biases are expensive**. A loss-averse investor
who panic-sells during drawdowns and waits to re-enter trails a rational
buy-and-hold investor by ~$90,000 over 10 years on a $100,000 initial investment.

---

### Academic grounding

| Concept | Paper |
|---|---|
| Prospect Theory (loss aversion) | Kahneman & Tversky (1979), *Econometrica* |
| Disposition effect | Shefrin & Statman (1985), *Journal of Finance* |
| Overconfidence & over-trading | Barber & Odean (2001), *QJE* |
| Fat-tailed returns | Cont (2001), *Quantitative Finance* |
| Meta-elliptical copula | McNeil, Frey & Embrechts (2015), *QRM* Ch.7 |
| Regime-switching | Hamilton (1989), *Econometrica* |


## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import expit

from src.data import load_market_data
from src.simulation import MonteCarloEngine, SimulationConfig, REGIME_CORRELATIONS
from src.investors import RationalInvestor, LossAverseInvestor, OverconfidentInvestor
from src.metrics import compute_metrics, compute_bias_costs, summary_table
from src.backtest import run_historical_backtest, backtest_summary_table
from src.calibration import compute_calibration_stats

plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#0D1117",
    "axes.facecolor":   "#161B22",
    "axes.edgecolor":   "#30363D",
    "axes.labelcolor":  "#E6EDF3",
    "xtick.color":      "#8B949E",
    "ytick.color":      "#8B949E",
    "grid.color":       "#21262D",
    "grid.linewidth":   0.5,
    "text.color":       "#E6EDF3",
    "font.family":      "monospace",
})

COLORS = {"Rational": "#4A9EFF", "Loss-Averse": "#FF7043", "Overconfident": "#AB47BC"}
print("Setup complete")


## 2. Real Market Data

We use daily closing prices for three ETFs downloaded from Stooq.com:
- **SPY** — SPDR S&P 500 ETF (US equities proxy, 60% weight)
- **EFA** — iShares MSCI EAFE ETF (international equities, 20%)
- **AGG** — iShares Core US Aggregate Bond ETF (bonds, 20%)

All distribution parameters are **fitted from actual returns** — nothing is hardcoded.


In [ ]:
md = load_market_data()
lr = md.log_returns

print(f"Period: {md.start_date} → {md.end_date}")
print(f"Observations: {md.n_obs:,} trading days ({md.n_obs/252:.1f} years)")
print()
print(f"{'Asset':<25} {'Ann. Return':>12} {'Ann. Vol':>10} {'t-dist ν':>10} {'Skew':>8}")
print("-" * 67)
for a in md.fitted_assets:
    print(f"{a.name:<25} {a.ann_return:>12.2%} {a.ann_vol:>10.2%} {a.nu:>10.2f} {a.skew:>8.3f}")

print()
print("Empirical correlation matrix:")
corr_df = pd.DataFrame(
    np.round(md.correlation, 3),
    index=[a.name.split("(")[0].strip() for a in md.fitted_assets],
    columns=[a.name.split("(")[0].strip() for a in md.fitted_assets],
)
print(corr_df.to_string())


## 3. Why Fat-Tailed Distributions Matter

Daily equity returns are **not normally distributed**. They exhibit:
- **Excess kurtosis** (fat tails): large losses occur far more often than a Gaussian predicts
- **Negative skew**: losses cluster; gains are more spread out
- **Low degrees of freedom**: SPY fitted ν ≈ 2.4 vs. ∞ for normal

This section shows the empirical evidence and the quality of the t-distribution fit.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Return distributions: empirical vs fitted t-distribution vs normal", 
             fontsize=12, y=1.02)

asset_colors = ["#4A9EFF", "#AB47BC", "#3FB950"]

for i, (ax, asset, col) in enumerate(zip(axes, md.fitted_assets, lr.columns)):
    r = lr[col].values
    lo, hi = np.percentile(r, 0.5), np.percentile(r, 99.5)
    x = np.linspace(lo, hi, 300)
    
    # Empirical histogram
    ax.hist(r, bins=120, density=True, color=asset_colors[i], alpha=0.4,
            label="Empirical", range=(lo, hi))
    
    # Fitted t-distribution
    pdf_t = stats.t.pdf(x, df=asset.nu, loc=asset.mu, scale=asset.sigma)
    ax.plot(x, pdf_t, color=asset_colors[i], lw=2, label=f"t(ν={asset.nu:.1f})")
    
    # Normal distribution (same mean and std)
    pdf_n = stats.norm.pdf(x, loc=r.mean(), scale=r.std())
    ax.plot(x, pdf_n, color="white", lw=1.5, ls="--", alpha=0.6, label="Normal")
    
    ax.set_title(asset.name.split("(")[0].strip(), color=asset_colors[i])
    ax.set_xlabel("Daily log-return")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Annotate kurtosis
    kurt = lr[col].kurtosis()
    ax.text(0.05, 0.95, f"Excess kurtosis: {kurt:.1f}\n(normal = 0)",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round", facecolor="#161B22", alpha=0.8))

plt.tight_layout()
plt.savefig("../outputs/01_return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Key insight: the t-distribution's fat tails fit the empirical data far better")
print("than the normal distribution, especially in the crash-probability tails.")


## 4. Regime-Switching Correlations

A critical real-world phenomenon: **correlations are not constant**.

During market crises, equity-equity correlations spike (everything falls together)
and equity-bond correlations can reverse sign — from negative (flight to quality)
to positive (simultaneous liquidation). This was dramatically visible in 2022
when both stocks and bonds fell sharply as interest rates rose.

We model three regimes identified by realised equity volatility:


In [ ]:
# Plot rolling correlations with regime shading
spy_vol = lr.iloc[:,0].rolling(21).std() * np.sqrt(252)
p40, p75 = spy_vol.quantile(0.40), spy_vol.quantile(0.75)
rc = md.rolling_corr

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Regime analysis: rolling correlations and equity volatility", fontsize=12)

# Vol with regime shading
ax = axes[0]
ax.plot(spy_vol.index, spy_vol.values, color="#E6EDF3", lw=1, label="SPY 21d vol")
ax.axhline(p40, color="#4A9EFF", ls="--", lw=1, label=f"Calm/Normal ({p40:.1%})")
ax.axhline(p75, color="#FF7043", ls="--", lw=1, label=f"Normal/Crisis ({p75:.1%})")
ax.fill_between(spy_vol.index, 0, spy_vol, where=spy_vol < p40,
                color="#4A9EFF", alpha=0.15, label="Calm regime")
ax.fill_between(spy_vol.index, 0, spy_vol, where=spy_vol > p75,
                color="#FF7043", alpha=0.20, label="Crisis regime")
ax.set_ylabel("Annualised vol")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# SPY/EFA correlation
pair_cols = rc.columns.tolist()
colors_rc = ["#4A9EFF", "#FF7043", "#3FB950"]
for i, (ax, col, clr) in enumerate(zip(axes[1:], pair_cols[:2], colors_rc[:2])):
    ax.plot(rc.index, rc[col], color=clr, lw=1.2)
    ax.axhline(0, color="#8B949E", ls="--", lw=0.8)
    # Shade regime periods
    ax.fill_between(spy_vol.loc[rc.index].index, -1, 1,
                    where=spy_vol.loc[rc.index] < p40,
                    color="#4A9EFF", alpha=0.06)
    ax.fill_between(spy_vol.loc[rc.index].index, -1, 1,
                    where=spy_vol.loc[rc.index] > p75,
                    color="#FF7043", alpha=0.08)
    ax.set_ylabel(col.replace(" / ", "\n/ ").replace(" (SPY)", "").replace(" (EFA)", "").replace(" (AGG)", ""), fontsize=8)
    ax.set_ylim(-0.7, 1.1)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig("../outputs/02_regime_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nRegime correlation matrices:")
print(f"{'Regime':<10} {'SPY/EFA':>10} {'SPY/AGG':>10} {'EFA/AGG':>10}")
print("-" * 42)
for name, C in REGIME_CORRELATIONS.items():
    print(f"{name:<10} {C[0,1]:>10.3f} {C[0,2]:>10.3f} {C[1,2]:>10.3f}")


## 5. The Probabilistic Panic Model

A key upgrade from naïve hard-threshold models: the loss-averse investor's
panic probability is a **logistic function** of drawdown depth, not a cliff edge.

This matches Prospect Theory's value function — pain is non-linear in losses,
and the *marginal* pain of an additional 1% loss increases as losses deepen.

$$P(\text{panic} \mid DD_t) = \sigma\left(k \cdot (|DD_t| - \theta)\right)$$

where $\sigma$ is the logistic function, $k$ controls steepness, and $\theta$
is the drawdown at which P=50%.


In [ ]:
la = LossAverseInvestor(panic_threshold=-0.10)  # P=50% at -10% DD
curve = la.panic_curve_data()

dd_vals = np.array([-0.03, -0.05, -0.08, -0.10, -0.12, -0.15, -0.20, -0.25, -0.30])
probs   = expit(la.panic_k * (-dd_vals - la.panic_midpoint))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: panic curve
ax = axes[0]
dd_range = np.linspace(-0.35, 0, 300)
p_range  = expit(la.panic_k * (-dd_range - la.panic_midpoint))
ax.plot(dd_range * 100, p_range * 100, color="#FF7043", lw=2.5)
ax.axvline(-la.panic_midpoint * 100, color="#FF7043", ls="--", lw=1, alpha=0.6,
           label=f"P=50% at DD={-la.panic_midpoint:.0%}")
ax.axhline(50, color="#8B949E", ls=":", lw=0.8)

# Hard threshold for comparison
ax.axvline(-10, color="#8B949E", ls="--", lw=1.5, alpha=0.5, label="Old: hard threshold at -10%")

for dd, p in zip(dd_vals, probs):
    ax.scatter(dd*100, p*100, color="#FF7043", zorder=5, s=40)
    ax.annotate(f"{p:.0%}", (dd*100, p*100), textcoords="offset points",
                xytext=(5, 5), fontsize=7, color="#E6EDF3")

ax.set_xlabel("Drawdown from peak (%)")
ax.set_ylabel("P(panic on this day) (%)")
ax.set_title("Logistic panic probability curve", color="#FF7043")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(-37, 2)
ax.set_ylim(-5, 105)

# Right: Prospect Theory value function (stylised)
ax2 = axes[1]
x = np.linspace(-1, 1, 300)
# Kahneman-Tversky value function: v(x) = x^alpha if x>=0, -lambda * (-x)^beta if x<0
alpha, beta, lam = 0.88, 0.88, 2.25
v = np.where(x >= 0, x**alpha, -lam * (-x)**beta)
ax2.plot(x[x>=0]*100, v[x>=0], color="#4A9EFF", lw=2.5, label="Gains")
ax2.plot(x[x<0]*100, v[x<0], color="#FF7043", lw=2.5, label="Losses")
ax2.axhline(0, color="#8B949E", lw=0.8)
ax2.axvline(0, color="#8B949E", lw=0.8)
ax2.set_xlabel("Outcome (% vs reference point)")
ax2.set_ylabel("Subjective value")
ax2.set_title("Prospect Theory value function\n(Kahneman & Tversky 1979)", color="#E6EDF3")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.annotate("Loss aversion:\nslope steeper\nfor losses", 
             xy=(-0.3, -0.6), fontsize=9, color="#FF7043",
             bbox=dict(boxstyle="round", facecolor="#161B22", alpha=0.8))

plt.tight_layout()
plt.savefig("../outputs/03_panic_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Monte Carlo Simulation

With all parameters data-driven and the model correctly specified, we run
10,000 simulations of a 10-year portfolio for each investor archetype.


In [ ]:
np.random.seed(42)
config = SimulationConfig(
    n_simulations=10_000,
    n_years=10,
    initial_wealth=100_000,
    use_regime_switching=True,
    random_seed=42,
)

print("Running simulation...")
paths = MonteCarloEngine(market_data=md, config=config).generate_paths()

rational    = RationalInvestor().simulate(paths)
loss_averse = LossAverseInvestor(panic_threshold=-0.10).simulate(paths)
overconf    = OverconfidentInvestor().simulate(paths)

results = [rational, loss_averse, overconf]
m_list  = compute_bias_costs([compute_metrics(r, config) for r in results])

print(f"\n{'Investor':<15} {'Median Wealth':>14} {'Ann. Return':>12} {'Sharpe':>8} {'Bias Cost':>12}")
print("-" * 63)
for row in summary_table(m_list):
    print(f"{row['Investor Type']:<15} {row['Median Wealth']:>14} {row['Ann. Return']:>12} {row['Sharpe Ratio']:>8} {row['Bias Cost ($)']:>12}")


## 7. Results: Wealth Distribution & Bias Cost

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.30)

# ── Panel 1: Terminal wealth distributions ────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
for r in results:
    W = r.terminal_wealth
    lo, hi = np.percentile(W, 0.5), np.percentile(W, 99.5)
    bins = np.linspace(lo, hi, 120)
    counts, edges = np.histogram(W, bins=bins, density=True)
    centres = (edges[:-1] + edges[1:]) / 2
    ax1.fill_between(centres, counts, alpha=0.25, color=COLORS[r.name])
    ax1.plot(centres, counts, lw=2, color=COLORS[r.name], label=r.name)
    ax1.axvline(np.median(W), color=COLORS[r.name], ls="--", lw=1.2, alpha=0.8)
ax1.axvline(config.initial_wealth, color="#FFD700", ls="--", lw=1.5, label="Initial ($100k)")
ax1.set_xlabel("Terminal wealth ($)")
ax1.set_ylabel("Probability density")
ax1.set_title("Terminal wealth distribution after 10 years (10,000 simulations)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Panel 2: Median wealth paths ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
x   = np.linspace(0, config.n_years, results[0].wealth_paths.shape[1])
for r in results:
    W   = r.wealth_paths
    med = np.median(W, axis=0)
    p25 = np.percentile(W, 25, axis=0)
    p75 = np.percentile(W, 75, axis=0)
    ax2.fill_between(x, p25, p75, alpha=0.10, color=COLORS[r.name])
    ax2.plot(x, med, lw=2, color=COLORS[r.name], label=r.name)
ax2.set_xlabel("Years")
ax2.set_ylabel("Portfolio value ($)")
ax2.set_title("Median wealth path (± IQR band)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: Bias cost bar ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
non_rat = [m for m in m_list if m.name != "Rational"]
bars = ax3.bar(
    [m.name for m in non_rat],
    [m.bias_cost_vs_rational for m in non_rat],
    color=[COLORS[m.name] for m in non_rat],
    width=0.45,
)
for bar, m in zip(bars, non_rat):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f"${m.bias_cost_vs_rational:,.0f}\n({m.bias_cost_pct:.1%})",
             ha="center", va="bottom", fontsize=9, color=COLORS[m.name])
ax3.set_ylabel("Wealth gap vs Rational ($)")
ax3.set_title("Behavioral bias cost (median terminal wealth)")
ax3.grid(True, alpha=0.3, axis="y")

plt.suptitle("Behavioral Finance Monte Carlo — Key Results", fontsize=13, y=1.01)
plt.savefig("../outputs/04_main_results.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Backtest Validation

A crucial credibility test: run the Rational investor strategy on the **actual**
historical return sequence and verify the outcome falls within the simulated
distribution. If the simulation is well-calibrated, the actual outcome should
be somewhere between the 10th and 90th percentile.


In [ ]:
bt = run_historical_backtest(md, initial_wealth=config.initial_wealth)
terminal = rational.terminal_wealth
actual   = bt["wealth_path"].iloc[-1]
pct_rank = (terminal < actual).mean() * 100

print("Historical backtest results (60/20/20 portfolio):")
print(f"  Period:         {bt['start_date']} → {bt['end_date']}")
print(f"  Ann. return:    {bt['ann_return']:.2%}")
print(f"  Ann. volatility:{bt['ann_vol']:.2%}")
print(f"  Sharpe ratio:   {bt['sharpe']:.2f}")
print(f"  Max drawdown:   {bt['max_drawdown']:.2%}")
print(f"  Terminal wealth:{actual:,.0f}")
print(f"  Percentile rank:{pct_rank:.0f}th (in simulated distribution)")
print()
print(f"  The 2005-2024 period was an exceptional bull market.")
print(f"  An outcome at p{pct_rank:.0f} is consistent with a strong but not extreme run.")

# Plot actual path vs simulation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: path comparison
ax = axes[0]
x_sim  = np.linspace(0, config.n_years, rational.wealth_paths.shape[1])
W      = rational.wealth_paths
for plo, phi, alpha in [(5, 95, 0.07), (25, 75, 0.14)]:
    ax.fill_between(x_sim, np.percentile(W, plo, axis=0), np.percentile(W, phi, axis=0),
                    color="#4A9EFF", alpha=alpha, label=f"Sim p{plo}–p{phi}")
ax.plot(x_sim, np.median(W, axis=0), color="#4A9EFF", lw=2, label="Sim median")
x_hist = np.linspace(0, bt["n_years"], len(bt["wealth_path"]))
ax.plot(x_hist, bt["wealth_path"].values, color="#FFD700", lw=2.5, label="Actual history")
ax.axhline(config.initial_wealth, color="#8B949E", ls="--", lw=0.8)
ax.set_xlabel("Years")
ax.set_ylabel("Portfolio value ($)")
ax.set_title("Actual path vs simulated distribution")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: terminal distribution with actual marked
ax = axes[1]
lo_b, hi_b = np.percentile(terminal, 0.5), np.percentile(terminal, 99.5)
ax.hist(terminal, bins=100, density=True, color="#4A9EFF", alpha=0.4,
        range=(lo_b, hi_b), label="Simulated")
ax.axvline(actual, color="#FFD700", lw=2.5, label=f"Actual: ${actual:,.0f}")
ax.axvline(np.median(terminal), color="#4A9EFF", lw=1.5, ls="--", 
           label=f"Sim median: ${np.median(terminal):,.0f}")
ax.text(actual * 1.01, ax.get_ylim()[1] * 0.85,
        f"p{pct_rank:.0f} of\ndistribution", color="#FFD700", fontsize=9)
ax.set_xlabel("Terminal wealth ($)")
ax.set_ylabel("Density")
ax.set_title(f"Actual outcome at p{pct_rank:.0f} of simulated distribution")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("Backtest validation: simulation is well-calibrated", fontsize=12)
plt.tight_layout()
plt.savefig("../outputs/05_backtest_validation.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Conclusions & Limitations

### Key findings

1. **Behavioural biases are quantitatively large**. A loss-averse investor who
   panic-sells at meaningful drawdowns and waits to re-enter trails the rational
   investor by ~$90,000 over 10 years on a $100k portfolio — roughly 90% of
   the initial investment, compounded into a wealth gap.

2. **Fat tails matter**. Using a Student's t-distribution (ν≈2.4) instead of
   a normal distribution meaningfully changes the tail risk estimates. The 5th
   percentile of terminal wealth (VaR) shifts by thousands of dollars.

3. **Regime-switching correlations matter**. The equity/bond correlation flips
   from −0.28 in calm periods to +0.16 in crisis — a 0.44 swing. This affects
   diversification benefit estimates significantly.

4. **The 2005–2024 period was above-average**. The actual 60/20/20 portfolio
   landed at the 83rd percentile of the simulated distribution, consistent with
   an unusually strong equity bull market (driven by post-GFC recovery, QE,
   and the technology sector).

### Limitations

- **Static weights**: target allocation is fixed at 60/20/20. Real portfolios
  drift and are rebalanced with varying frequency.
- **No transaction costs for rational investor**: the loss-averse and
  overconfident investors pay costs; the rational investor does not (conservative bias).
- **Correlation regime identification**: we use a simple vol threshold. A formal
  Hidden Markov Model (Hamilton 1989) would be more rigorous.
- **No momentum or factor exposures**: actual ETF returns include factor tilts
  not captured by the marginal distributions alone.
- **Two-decade sample**: 2005–2024 covers only two full market cycles. 
  Longer data (going back to the 1920s via Fama-French factors) would give 
  more stable parameter estimates.

### Potential extensions

- Bootstrap historical returns instead of parametric simulation
- Add a fourth archetype: the "Disposition Effect" investor (sells winners too
  early, holds losers too long — Shefrin & Statman 1985)
- Add factor models (Fama-French 3-factor) to the return generation
- Formal HMM regime identification (Hamilton 1989)


## References

- Barber, B.M. & Odean, T. (2001). Boys will be boys: Gender, overconfidence, 
  and common stock investment. *Quarterly Journal of Economics*, 116(1), 261–292.
  
- Barberis, N., Huang, M. & Santos, T. (2001). Prospect theory and asset prices. 
  *Quarterly Journal of Economics*, 116(1), 1–53.

- Cont, R. (2001). Empirical properties of asset returns: Stylized facts and 
  statistical issues. *Quantitative Finance*, 1(2), 223–236.
  
- Hamilton, J.D. (1989). A new approach to the economic analysis of nonstationary 
  time series and the business cycle. *Econometrica*, 57(2), 357–384.

- Joe, H. (2014). *Dependence Modeling with Copulas*. CRC Press.

- Kahneman, D. & Tversky, A. (1979). Prospect theory: An analysis of decision 
  under risk. *Econometrica*, 47(2), 263–291.

- Malmendier, U. & Tate, G. (2005). CEO overconfidence and corporate investment. 
  *Journal of Finance*, 60(6), 2661–2700.

- Markowitz, H. (1952). Portfolio selection. *Journal of Finance*, 7(1), 77–91.

- McNeil, A.J., Frey, R. & Embrechts, P. (2015). *Quantitative Risk Management: 
  Concepts, Techniques and Tools* (Rev. ed.). Princeton University Press.

- Odean, T. (1998). Are investors reluctant to realise their losses? 
  *Journal of Finance*, 53(5), 1775–1798.

- Shefrin, H. & Statman, M. (1985). The disposition to sell winners too early 
  and ride losers too long: Theory and evidence. *Journal of Finance*, 40(3), 777–790.
